In [41]:
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate as sp
import math as mt
from scipy.integrate import solve_ivp

output_dir = "/home/max/Documents/private/Uni/masterthesis/paper/plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def Gamma(d): #Define Gamma function as numerical integral
    result, _ = sp.quad(lambda t: t**(d-1)*np.exp(-t), 0, np.inf) #returns only numerical value, _ "deletes" the error value from sp.quad
    return result

In [50]:
#coupled deq derived from wetterich eq
def flow(k, S, d, rho): #del_t Gamma_k = ...
    mq , l = S #mq = m², l = lambda

    prefactor1 = (12*k**(d+1))/((4*np.pi)**(d/2)*Gamma(d/2 + 1))
    prefactor2 = (2*k**(d+1))/((4*np.pi)**(d/2)*Gamma(d/2 + 1))
 
    dldk = (prefactor1*((l**2) / (18*(k**2 +mq + l*rho)**3) + (2*l**2) / (9*(k**2 +mq + 3*l*rho)**3))
      )  #dlambda/dk
    
    dmqdk = (prefactor2*(-l / (6*(k**2 + mq + l*rho)**2) - l / (3*(k**2 + mq + 3*l*rho)**2 )) - rho*dldk/6
      )  #dm²/dk

    return [dmqdk, dldk]

In [51]:
#initial conditions
d = 3.0
rho = 50.0
k0 = 1000.0
k_end = 0.0

l0_list=[0.0001, 0.001, 0.01, 0.1, 1, 10, 20, 100]
mq0_list=[-16, 16]

k_eval = np.linspace(k0, k_end, 1000)


In [57]:
# Schleife über die beiden m0²-Werte
for mq0 in mq0_list:
    # Vorab alle Lösungen für die 8 l0-Werte berechnen
    solutions = {}
    for l0 in l0_list:
        S0 = [mq0, l0]
        sol = solve_ivp(flow, t_span=(k0, k_end), y0=S0, t_eval=k_eval, method='RK45', args=(d, rho))
        if sol.success:
            solutions[l0] = sol
        else:
            print(f"Integration fehlgeschlagen für mq0={mq0}, l0={l0}: {sol.message}")

    # --- PLOT 1: Nur m^2(k) in Kachelansicht ---
    fig_mq, axes_mq = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
    axes_mq_flat = axes_mq.flatten()
    
    for idx, l0 in enumerate(l0_list):
        ax = axes_mq_flat[idx]
        if l0 in solutions:
            sol = solutions[l0]
            ax.plot(sol.t, sol.y[0], color='tab:blue', lw=1.5)
            ax.set_ylabel(r'$m^2(k)$', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.set_xlim(k0, k_end)
        else:
            ax.text(0.5, 0.5, "Integration failed", transform=ax.transAxes, ha='center')
        ax.set_title(f"$\\lambda_0 = {l0}$", fontsize=10, fontweight='bold')

    for ax in axes_mq[1, :]:
        ax.set_xlabel(r'Scale $k$ (MeV)', fontsize=9)

    fig_mq.suptitle(f"RG-Flow ($m^2(k)$) for $m_0^2 = {mq0}$ MeV² ($d={d}, \\rho={rho}$)", fontsize=14)
    plt.tight_layout()
    
    file_path_mq = os.path.join(output_dir, f"rg_flow_grid_mq0_{int(mq0)}_mq.png")
    fig_mq.savefig(file_path_mq, dpi=300)
    plt.close(fig_mq)
    print(f"m^2 Kachel-Plot gespeichert: {file_path_mq}")

    # --- PLOT 2: Nur \lambda(k) in Kachelansicht ---
    fig_l, axes_l = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
    axes_l_flat = axes_l.flatten()
    
    for idx, l0 in enumerate(l0_list):
        ax = axes_l_flat[idx]
        if l0 in solutions:
            sol = solutions[l0]
            ax.plot(sol.t, sol.y[1], color='tab:orange', lw=1.5)
            ax.set_ylabel(r'$\lambda(k)$', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.set_xlim(k0, k_end)
        else:
            ax.text(0.5, 0.5, "Integration failed", transform=ax.transAxes, ha='center')
        ax.set_title(f"$\\lambda_0 = {l0}$", fontsize=10, fontweight='bold')

    for ax in axes_l[1, :]:
        ax.set_xlabel(r'Scale $k$ (MeV)', fontsize=9)

    fig_l.suptitle(f"RG-Flow ($\\lambda(k)$) for $m_0^2 = {mq0}$ MeV² ($d={d}, \\rho={rho}$)", fontsize=14)
    plt.tight_layout()
    
    file_path_l = os.path.join(output_dir, f"rg_flow_grid_mq0_{int(mq0)}_lambda.png")
    fig_l.savefig(file_path_l, dpi=300)
    plt.close(fig_l)
    print(f"lambda Kachel-Plot gespeichert: {file_path_l}")

m^2 Kachel-Plot gespeichert: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_-16_mq.png
lambda Kachel-Plot gespeichert: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_-16_lambda.png
m^2 Kachel-Plot gespeichert: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_16_mq.png
lambda Kachel-Plot gespeichert: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_16_lambda.png
